In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
from itertools import product

from sklearn.model_selection import train_test_split
import joblib

In [6]:
# ============================================================
# 2. SET PATHS
# ============================================================
DATA_PATH = Path(r"D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_primary_only_v1.csv")
BASE_DIR = Path(r"D:\khoane\MAHE\Project\model\phase1_protein_baseline")
SPLIT_DIR = BASE_DIR / "splits"
FEATURE_DIR = BASE_DIR / "features"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"

for folder in [SPLIT_DIR, FEATURE_DIR, RESULT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1820, 30)


,gene_id,gene_symbol,gene_name,uniprot_accession,uniprot_entry_name,sequence,sequence_length_fasta,label,label_name,dataset_role,...,uniprot_to_ensg_status,representative_sequence_rule,length_bin,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand
0,ENSG00000161618,ALDH16A1,ALDH16A1,Q8IZ83,A16A1_HUMAN,MAATRAGPRAREIFTSLEYGPVPESHACALAWLDTQDRCLGHYVNG...,802.0,0,T2D_negative,negative,...,one_uniprot_to_one_ensg,NaN,"(683.2, 852.6]",ENSG00000161618,ALDH16A1,protein_coding,19,49453146.0,49471052.0,1.0
1,ENSG00000224420,ADM5,ADM5,C9JUS6,ADM5_HUMAN,MTIHILILLLLLAFSAQGDLDTAARRGQHQVPQHRGHVCYLGVCRT...,153.0,0,T2D_negative,negative,...,one_uniprot_to_one_ensg,NaN,"(40.999, 210.0]",ENSG00000224420,ADM5,protein_coding,19,49688953.0,49690575.0,1.0
2,ENSG00000106113,CRHR2,corticotropin releasing hormone receptor 2,Q13324,CRFR2_HUMAN,MDAALLHSLLEANCSLALAEELLLDGWGPPLDPEGPYSYCNTTLDQ...,411.0,1,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000106113,CRHR2,protein_coding,7,30651444.0,30700129.0,-1.0
3,ENSG00000165392,WRN,WRN RecQ like helicase,Q14191,WRN_HUMAN,MSEKKLETTAQQRKCPEWMNVQNKRCAVEERKACVRKSVFEDDLPF...,1432.0,1,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000165392,WRN,protein_coding,8,31033779.0,31176138.0,1.0
4,ENSG00000131196,NFATC1,nuclear factor of activated T cells 1,O95644,NFAC1_HUMAN,MPSTSFPVPSKFPLGPAAAVFGRGETLGPAPRAGGTMKSAEEEHYG...,943.0,1,T2D_positive,positive,...,one_uniprot_to_one_ensg,single_selected_sequence,NaN,ENSG00000131196,NFATC1,protein_coding,18,79395856.0,79529325.0,1.0


In [4]:
# ============================================================
# 4. INSPECT COLUMNS
# ============================================================

print("Columns:")
for col in df.columns:
    print("-", col)

print("\nDataset info:")
df.info()

Columns:
- gene_id
- gene_symbol
- gene_name
- uniprot_accession
- uniprot_entry_name
- sequence
- sequence_length_fasta
- label
- label_name
- dataset_role
- class_source
- classification_dataset_version
- association_score
- positive_tier
- label_source
- label_rule
- label_version
- negative_sample_version
- negative_sampling_rule
- mapping_route
- uniprot_to_ensg_status
- representative_sequence_rule
- length_bin
- ensembl_gene_id
- ensembl_gene_name
- gene_type
- chromosome_or_scaffold
- gene_start_bp
- gene_end_bp
- strand

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1820 entries, 0 to 1819
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   gene_id                         1820 non-null   object 
 1   gene_symbol                     1819 non-null   object 
 2   gene_name                       1819 non-null   object 
 3   uniprot_accession               1

In [5]:
# ============================================================
# 5. REQUIRED COLUMNS CHECK
# ============================================================

required_cols = [
    "gene_id",
    "gene_symbol",
    "uniprot_accession",
    "sequence",
    "sequence_length_fasta",
    "label"
]

missing_cols = [col for col in required_cols if col not in df.columns]

print("Missing required columns:", missing_cols)

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

Missing required columns: []


In [6]:
# ============================================================
# 6. LABEL DISTRIBUTION CHECK
# ============================================================

print("Label counts:")
print(df["label"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts(normalize=True))

Label counts:
label
0    910
1    910
Name: count, dtype: int64

Label distribution:
label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [7]:
# ============================================================
# 7. MISSING VALUE CHECK
# ============================================================

print("Missing values in required columns:")
print(df[required_cols].isna().sum())

if df["sequence"].isna().sum() > 0:
    raise ValueError("There are missing protein sequences.")

if df["label"].isna().sum() > 0:
    raise ValueError("There are missing labels.")

Missing values in required columns:
gene_id                  0
gene_symbol              1
uniprot_accession        0
sequence                 0
sequence_length_fasta    0
label                    0
dtype: int64


In [11]:
missing_gene_symbol_rows = df[df["gene_symbol"].isna()]

display(
    missing_gene_symbol_rows[
        ["gene_id", "gene_symbol", "uniprot_accession", "label", "sequence_length_fasta"]
    ]
)

,gene_id,gene_symbol,uniprot_accession,label,sequence_length_fasta
1635,ENSG00000283599,NaN,A0A1B0GWI6,0,518.0


In [12]:
df["gene_symbol"] = df["gene_symbol"].fillna("UNKNOWN")

In [8]:
# ============================================================
# 8. DUPLICATE CHECK
# ============================================================

duplicate_gene = df["gene_id"].duplicated().sum()
duplicate_uniprot = df["uniprot_accession"].duplicated().sum()
duplicate_sequence = df["sequence"].duplicated().sum()

print("Duplicate gene_id:", duplicate_gene)
print("Duplicate uniprot_accession:", duplicate_uniprot)
print("Duplicate sequence:", duplicate_sequence)

if duplicate_sequence > 0:
    print("Warning: duplicate protein sequences found. Check possible leakage risk.")

Duplicate gene_id: 0
Duplicate uniprot_accession: 0
Duplicate sequence: 0


In [9]:
# ============================================================
# 9. SEQUENCE LENGTH CHECK
# ============================================================

df["computed_sequence_length"] = df["sequence"].astype(str).str.len()

length_mismatch = df[df["computed_sequence_length"] != df["sequence_length_fasta"]]

print("Sequence length summary:")
display(df["computed_sequence_length"].describe())

print("Length mismatch rows:", len(length_mismatch))

if len(length_mismatch) > 0:
    display(length_mismatch[[
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "sequence_length_fasta",
        "computed_sequence_length"
    ]].head())

Sequence length summary:


count     1820.000000
mean       767.783516
std       1015.463258
min         41.000000
25%        353.000000
50%        561.000000
75%        941.000000
max      34350.000000
Name: computed_sequence_length, dtype: float64

Length mismatch rows: 0


In [14]:
# ============================================================
# 10. AMINO ACID VALIDITY CHECK
# ============================================================

STANDARD_AAS = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AAS)

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq

df["sequence_clean"] = df["sequence"].apply(clean_sequence)
df["sequence_clean_length"] = df["sequence_clean"].str.len()

df["removed_non_standard_count"] = (
    df["sequence"].str.len() - df["sequence_clean"].str.len()
)

df[df["removed_non_standard_count"] > 0][
    [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "label",
        "sequence_length_fasta",
        "sequence_clean_length",
        "removed_non_standard_count",
        "non_standard_aas"
    ]
]

,gene_id,gene_symbol,uniprot_accession,label,sequence_length_fasta,sequence_clean_length,removed_non_standard_count,non_standard_aas
937,ENSG00000179918,SEPHS2,Q99611,0,448.0,447,1,{U}


In [24]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

In [25]:
train_df

,gene_id,gene_symbol,gene_name,uniprot_accession,uniprot_entry_name,sequence,sequence_length_fasta,label,label_name,dataset_role,...,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,computed_sequence_length,non_standard_aas,sequence_clean,sequence_clean_length,removed_non_standard_count
1189,ENSG00000205155,PSENEN,PSENEN,Q9NZ42,PEN2_HUMAN,MNLERVSNEEKLNLCRKYYLGGFAFLPFLWLVNIFWFFREAFLVPA...,101.0,0,T2D_negative,negative,...,protein_coding,19,35745070.0,35747519.0,1.0,101,{},MNLERVSNEEKLNLCRKYYLGGFAFLPFLWLVNIFWFFREAFLVPA...,101,0
1771,ENSG00000164530,PI16,peptidase inhibitor 16,Q6UXB8,PI16_HUMAN,MHGSCSFLMLLLPLLLLLVATTGPVGALTDEEKRLMVELHNLYRAQ...,463.0,1,T2D_positive,positive,...,protein_coding,6,36948263.0,36965993.0,1.0,463,{},MHGSCSFLMLLLPLLLLLVATTGPVGALTDEEKRLMVELHNLYRAQ...,463,0
643,ENSG00000143167,GPA33,GPA33,Q99795,GPA33_HUMAN,MVGKMWPVLWTLCAVRVTVDAISVETPQDVLRASQGKSVTLPCTYH...,319.0,0,T2D_negative,negative,...,protein_coding,1,167048289.0,167166479.0,-1.0,319,{},MVGKMWPVLWTLCAVRVTVDAISVETPQDVLRASQGKSVTLPCTYH...,319,0
1734,ENSG00000137691,CFAP300,CFAP300,Q9BRQ4,CF300_HUMAN,MATGELGDLGGYYFRFLPQKTFQSLSSKEITSRLRQWSMLGRIKAQ...,267.0,0,T2D_negative,negative,...,protein_coding,11,102047435.0,102084554.0,1.0,267,{},MATGELGDLGGYYFRFLPQKTFQSLSSKEITSRLRQWSMLGRIKAQ...,267,0
1229,ENSG00000095981,KCNK16,potassium two pore domain channel subfamily K ...,Q96T55,KCNKG_HUMAN,MPSAGLCSCWGGRVLPLLLAYVCYLLLGATIFQLLERQAEAQSRDQ...,309.0,1,T2D_positive,positive,...,protein_coding,6,39314698.0,39322968.0,-1.0,309,{},MPSAGLCSCWGGRVLPLLLAYVCYLLLGATIFQLLERQAEAQSRDQ...,309,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,ENSG00000188626,GOLGA8M,GOLGA8M,H3BSY2,GOG8M_HUMAN,MAEETQHNKLAAAKKKLKEYWQKNSPRVPAGANRNRKTNGSIPQTA...,632.0,0,T2D_negative,negative,...,protein_coding,15,28698583.0,28738384.0,-1.0,632,{},MAEETQHNKLAAAKKKLKEYWQKNSPRVPAGANRNRKTNGSIPQTA...,632,0
430,ENSG00000228672,PROB1,PROB1,E7EW31,PROB1_HUMAN,MLTALAPPALPGIPRQLPTAPARRQDSSGSSGSYYTAPGSPEPPDV...,1015.0,0,T2D_negative,negative,...,protein_coding,5,139390592.0,139395104.0,-1.0,1015,{},MLTALAPPALPGIPRQLPTAPARRQDSSGSSGSYYTAPGSPEPPDV...,1015,0
717,ENSG00000165476,REEP3,receptor accessory protein 3,Q6NUK4,REEP3_HUMAN,MVSWMISRAVVLVFGMLYPAYYSYKAVKTKNVKEYVRWMMYWIVFA...,255.0,1,T2D_positive,positive,...,protein_coding,10,63521397.0,63625128.0,1.0,255,{},MVSWMISRAVVLVFGMLYPAYYSYKAVKTKNVKEYVRWMMYWIVFA...,255,0
1310,ENSG00000113761,ZNF346,ZNF346,Q9UL40,ZN346_HUMAN,MEYPAPATVQAADGGAAGPYSSSELLEGQEPDGVRFDRERARRLWE...,294.0,0,T2D_negative,negative,...,protein_coding,5,177022696.0,177081201.0,1.0,294,{},MEYPAPATVQAADGGAAGPYSSSELLEGQEPDGVRFDRERARRLWE...,294,0


In [26]:
# ============================================================
# 2. STANDARD AMINO ACIDS
# ============================================================

STANDARD_AAS = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AAS)

DIPEPTIDES = ["".join(pair) for pair in product(STANDARD_AAS, repeat=2)]

print("Number of standard amino acids:", len(STANDARD_AAS))
print("Number of dipeptides:", len(DIPEPTIDES))

Number of standard amino acids: 20
Number of dipeptides: 400


In [28]:
# ============================================================
# 3. CLEAN SEQUENCE FUNCTION
# ============================================================

def clean_sequence(seq):
    """
    Keep only the 20 standard amino acids.

    This is used because handcrafted AAC/DPC features are defined over
    the standard 20 amino acids only.
    """
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq

In [29]:
# ============================================================
# 4. ENSURE EACH SPLIT HAS sequence_clean
# ============================================================

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

if "sequence_clean" not in train_df.columns:
    train_df["sequence_clean"] = train_df["sequence"].apply(clean_sequence)

if "sequence_clean" not in val_df.columns:
    val_df["sequence_clean"] = val_df["sequence"].apply(clean_sequence)

if "sequence_clean" not in test_df.columns:
    test_df["sequence_clean"] = test_df["sequence"].apply(clean_sequence)


In [30]:
# ============================================================
# 5. FEATURE GROUP: LENGTH
# ============================================================

def extract_length_feature(seq):
    """
    Protein length feature.
    """
    seq = clean_sequence(seq)

    return {
        "protein_length": len(seq)
    }

In [31]:
# ============================================================
# 6. FEATURE GROUP: AAC
# ============================================================

def extract_aac(seq):
    """
    Amino Acid Composition.

    For each amino acid:
    AAC_A = count(A) / sequence_length
    """
    seq = clean_sequence(seq)
    length = len(seq)

    features = {}

    for aa in STANDARD_AAS:
        if length > 0:
            features[f"AAC_{aa}"] = seq.count(aa) / length
        else:
            features[f"AAC_{aa}"] = 0.0

    return features

In [32]:
# ============================================================
# 7. FEATURE GROUP: DPC
# ============================================================

def extract_dpc(seq):
    """
    Dipeptide Composition.

    For each dipeptide:
    DPC_AA = count(AA) / total_number_of_dipeptides
    """
    seq = clean_sequence(seq)
    total_pairs = len(seq) - 1

    features = {f"DPC_{dp}": 0.0 for dp in DIPEPTIDES}

    if total_pairs <= 0:
        return features

    for i in range(total_pairs):
        pair = seq[i:i+2]

        if pair in DIPEPTIDES:
            features[f"DPC_{pair}"] += 1

    for dp in DIPEPTIDES:
        features[f"DPC_{dp}"] = features[f"DPC_{dp}"] / total_pairs

    return features

In [33]:
# ============================================================
# 8. FEATURE GROUP: PHYSCHEM
# ============================================================

AA_GROUPS = {
    "hydrophobic": set("AVILMFWY"),
    "polar": set("STNQC"),
    "positive_charge": set("KRH"),
    "negative_charge": set("DE"),
    "special": set("GP")
}

def extract_physchem(seq):
    """
    Simple physicochemical grouped features.
    """
    seq = clean_sequence(seq)
    length = len(seq)

    features = {}

    if length == 0:
        features["frac_hydrophobic"] = 0.0
        features["frac_polar"] = 0.0
        features["frac_positive_charge"] = 0.0
        features["frac_negative_charge"] = 0.0
        features["frac_special"] = 0.0
        features["charge_balance"] = 0.0
        return features

    hydrophobic_count = sum(seq.count(aa) for aa in AA_GROUPS["hydrophobic"])
    polar_count = sum(seq.count(aa) for aa in AA_GROUPS["polar"])
    positive_count = sum(seq.count(aa) for aa in AA_GROUPS["positive_charge"])
    negative_count = sum(seq.count(aa) for aa in AA_GROUPS["negative_charge"])
    special_count = sum(seq.count(aa) for aa in AA_GROUPS["special"])

    features["frac_hydrophobic"] = hydrophobic_count / length
    features["frac_polar"] = polar_count / length
    features["frac_positive_charge"] = positive_count / length
    features["frac_negative_charge"] = negative_count / length
    features["frac_special"] = special_count / length
    features["charge_balance"] = (positive_count - negative_count) / length

    return features

In [34]:
# ============================================================
# 9. COMBINE ALL FEATURE GROUPS
# ============================================================

def extract_all_features(seq):
    """
    Combine:
    - protein length
    - AAC
    - DPC
    - physicochemical grouped features

    Expected total:
    1 + 20 + 400 + 6 = 427 features
    """
    features = {}

    features.update(extract_length_feature(seq))
    features.update(extract_aac(seq))
    features.update(extract_dpc(seq))
    features.update(extract_physchem(seq))

    return features

In [35]:
# ============================================================
# 10. BUILD FEATURE MATRIX
# ============================================================

def build_feature_matrix(input_df, sequence_col="sequence_clean"):
    """
    Convert a dataframe of protein sequences into numeric features.
    """
    feature_dicts = input_df[sequence_col].apply(extract_all_features)

    feature_df = pd.DataFrame(
        feature_dicts.tolist(),
        index=input_df.index
    )

    return feature_df


In [36]:
# ============================================================
# 11. BUILD X AND y FOR EACH SPLIT
# ============================================================

X_train = build_feature_matrix(train_df, sequence_col="sequence_clean")
X_val = build_feature_matrix(val_df, sequence_col="sequence_clean")
X_test = build_feature_matrix(test_df, sequence_col="sequence_clean")

y_train = train_df["label"].astype(int)
y_val = val_df["label"].astype(int)
y_test = test_df["label"].astype(int)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

display(X_train.head())

X_train shape: (1274, 427)
X_val shape: (273, 427)
X_test shape: (273, 427)
y_train shape: (1274,)
y_val shape: (273,)
y_test shape: (273,)


,protein_length,AAC_A,AAC_C,AAC_D,AAC_E,AAC_F,AAC_G,AAC_H,AAC_I,AAC_K,...,DPC_YT,DPC_YV,DPC_YW,DPC_YY,frac_hydrophobic,frac_polar,frac_positive_charge,frac_negative_charge,frac_special,charge_balance
1189,101,0.049505,0.009901,0.009901,0.049505,0.108911,0.069307,0.000000,0.069307,0.029703,...,0.010000,0.010000,0.0,0.010000,0.554455,0.178218,0.089109,0.059406,0.118812,0.029703
1771,463,0.097192,0.025918,0.034557,0.084233,0.021598,0.071274,0.034557,0.017279,0.043197,...,0.002165,0.000000,0.0,0.000000,0.360691,0.241901,0.110151,0.118790,0.168467,-0.008639
643,319,0.047022,0.034483,0.050157,0.087774,0.006270,0.056426,0.012539,0.068966,0.037618,...,0.000000,0.003145,0.0,0.003145,0.338558,0.307210,0.106583,0.137931,0.109718,-0.031348
1734,267,0.033708,0.029963,0.071161,0.052434,0.074906,0.052434,0.026217,0.063670,0.067416,...,0.000000,0.000000,0.0,0.003759,0.411985,0.232210,0.142322,0.123596,0.089888,0.018727
1229,309,0.080906,0.025890,0.022654,0.035599,0.064725,0.097087,0.016181,0.048544,0.022654,...,0.003247,0.006494,0.0,0.000000,0.485437,0.213592,0.093851,0.058252,0.148867,0.035599


In [37]:
# ============================================================
# FEATURE MATRIX SANITY CHECKS
# ============================================================

expected_n_features = 1 + 20 + 400 + 6

print("Expected number of features:", expected_n_features)
print("Actual number of features:", X_train.shape[1])

assert X_train.shape[1] == expected_n_features
assert X_val.shape[1] == expected_n_features
assert X_test.shape[1] == expected_n_features

assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_val:", X_val.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

print("Infinite values in X_train:", np.isinf(X_train.to_numpy()).sum())
print("Infinite values in X_val:", np.isinf(X_val.to_numpy()).sum())
print("Infinite values in X_test:", np.isinf(X_test.to_numpy()).sum())

print("Feature extraction sanity checks passed.")

Expected number of features: 427
Actual number of features: 427
Missing values in X_train: 0
Missing values in X_val: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_val: 0
Infinite values in X_test: 0
Feature extraction sanity checks passed.


In [38]:
# ============================================================
# AAC / DPC SUM CHECKS
# ============================================================

aac_cols = [col for col in X_train.columns if col.startswith("AAC_")]
dpc_cols = [col for col in X_train.columns if col.startswith("DPC_")]

X_train["AAC_sum_check"] = X_train[aac_cols].sum(axis=1)
X_train["DPC_sum_check"] = X_train[dpc_cols].sum(axis=1)

print("AAC sum summary:")
display(X_train["AAC_sum_check"].describe())

print("DPC sum summary:")
display(X_train["DPC_sum_check"].describe())

# Remove check columns after inspection
X_train = X_train.drop(columns=["AAC_sum_check", "DPC_sum_check"])

AAC sum summary:


count    1.274000e+03
mean     1.000000e+00
std      1.280716e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: AAC_sum_check, dtype: float64

DPC sum summary:


count    1.274000e+03
mean     1.000000e+00
std      1.946338e-15
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: DPC_sum_check, dtype: float64

In [ ]:
# ============================================================
# SAVE FEATURE MATRICES WITH METADATA
# ============================================================

def attach_metadata(split_df, X, y):
    """
    Attach metadata columns before feature matrix.
    """
    output_df = X.copy()

    output_df.insert(0, "label", y.values)
    output_df.insert(0, "sequence_clean_length", split_df["sequence_clean"].str.len().values)
    output_df.insert(0, "uniprot_accession", split_df["uniprot_accession"].values)
    output_df.insert(0, "gene_symbol", split_df["gene_symbol"].values)
    output_df.insert(0, "gene_id", split_df["gene_id"].values)

    return output_df


train_features_out = attach_metadata(train_df, X_train, y_train)
val_features_out = attach_metadata(val_df, X_val, y_val)
test_features_out = attach_metadata(test_df, X_test, y_test)

train_features_out.to_csv(
    FEATURE_DIR / "protein_features_full_train_v1.csv",
    index=False
)

val_features_out.to_csv(
    FEATURE_DIR / "protein_features_full_val_v1.csv",
    index=False
)

test_features_out.to_csv(
    FEATURE_DIR / "protein_features_full_test_v1.csv",
    index=False
)

print("Saved feature files:")
print(FEATURE_DIR / "protein_features_full_train_v1.csv")
print(FEATURE_DIR / "protein_features_full_val_v1.csv")
print(FEATURE_DIR / "protein_features_full_test_v1.csv")

Saved feature files:
D:\khoane\MAHE\Project\model\phase1_protein_baseline\features\protein_features_full_train_v1.csv
D:\khoane\MAHE\Project\model\phase1_protein_baseline\features\protein_features_full_val_v1.csv
D:\khoane\MAHE\Project\model\phase1_protein_baseline\features\protein_features_full_test_v1.csv
